# Homework 4: Pretrained Transformers

---

**Student Name:** Swesan Pathmanathan

**Student ID:** 400367859

**CodaBench Username:** swesan

**Assigned Task (from Group Assignments A4.txt):** Group 28 - CodaBench URL: https://www.codabench.org/competitions/14904/

**Self-Chosen Secondary Task:** Group *(number)* - CodaBench URL: *(paste URL here)*

---

### AI Tool Usage Declaration

*(Describe AI tool usage here, or write "No AI tools were used.")*

---
## 0. Setup & Dependencies

Install and import all libraries you need here.

In [2]:
# Install required packages
# If you see: `No module named 'huggingface_hub._snapshot_download'`
# upgrade `huggingface_hub` + `transformers` together, then restart the kernel.
!pip install -q -U huggingface_hub transformers datasets accelerate evaluate
!pip install -q scikit-learn torch sentencepiece protobuf scipy tqdm

# Core imports
import os
import random
import numpy as np
import pandas as pd

# HuggingFace
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline,
)

# Torch
import torch

# Scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    jaccard_score,
    roc_auc_score,
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -----------------------------
# Multi-label evaluation helpers
# -----------------------------

def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))


def subset_accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    # Strict exact match of the full label set per example
    return float((y_true == y_pred).all(axis=1).mean())


def multilabel_metrics(
    y_true: np.ndarray,
    y_prob: np.ndarray | None = None,
    y_pred: np.ndarray | None = None,
    threshold: float = 0.5,
) -> dict:
    if y_pred is None:
        if y_prob is None:
            raise ValueError("Provide y_prob or y_pred")
        y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "subset_acc": subset_accuracy(y_true, y_pred),
        "micro_f1": float(f1_score(y_true, y_pred, average="micro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        # IoU/Jaccard: average over samples (partial credit per instance)
        "jaccard": float(jaccard_score(y_true, y_pred, average="samples", zero_division=0)),
    }

    if y_prob is not None:
        try:
            # Multi-label ROC-AUC (macro over labels)
            metrics["roc_auc_macro"] = float(roc_auc_score(y_true, y_prob, average="macro"))
        except ValueError:
            metrics["roc_auc_macro"] = None

    return metrics


def print_metrics(title: str, metrics: dict) -> None:
    printable = ", ".join(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}" for k, v in metrics.items())
    print(f"{title}: {printable}")


def trainer_compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = sigmoid(logits)
    y_true = labels.astype(int)
    return multilabel_metrics(y_true=y_true, y_prob=probs, threshold=0.5)


def best_threshold_on_val(y_true: np.ndarray, y_prob: np.ndarray, metric: str = "micro_f1") -> float:
    # Simple global threshold search
    candidates = np.linspace(0.1, 0.9, 17)
    best_t, best_v = 0.5, -1.0
    for t in candidates:
        m = multilabel_metrics(y_true=y_true, y_prob=y_prob, threshold=float(t))
        v = m.get(metric)
        if v is not None and v > best_v:
            best_v, best_t = v, float(t)
    return best_t

Error importing huggingface_hub._snapshot_download: No module named 'huggingface_hub._snapshot_download'


ModuleNotFoundError: No module named 'huggingface_hub._snapshot_download'

---
# PART 1 — Assigned Task

Complete **all** steps (#3.1 Fine-tuning, #3.2 Zero-shot, #3.3 Baselines) for your **assigned** task.

## 1.1 Dataset Loading & Exploration

In [ ]:
# Load Group 28 (Genre no Jutsu) dataset from extracted CSVs
# Paths (relative to repo root)
TRAIN_PATH = "assigned_task/data/raw/dev/training_genre-no-jutsu.csv"
VAL_X_PATH = "assigned_task/data/raw/dev/validation_genre-no-jutsu.csv"
VAL_Y_PATH = "assigned_task/data/raw/dev/validation_labels_genre-no-jutsu.csv"

train_df = pd.read_csv(TRAIN_PATH, encoding="utf-8")
val_x_df = pd.read_csv(VAL_X_PATH, encoding="utf-8")
val_y_df = pd.read_csv(VAL_Y_PATH, encoding="utf-8")

# Validation labels are provided separately; join on anime_id
val_df = val_x_df.merge(val_y_df, on="anime_id", how="inner", validate="one_to_one")

required_train_cols = {"anime_id", "Name", "English name", "Synopsis", "Genres"}
required_val_cols = {"anime_id", "Name", "English name", "Synopsis", "Genres"}

missing_train = required_train_cols - set(train_df.columns)
missing_val = required_val_cols - set(val_df.columns)
if missing_train:
    raise ValueError(f"Train CSV missing columns: {sorted(missing_train)}")
if missing_val:
    raise ValueError(f"Validation CSV missing columns after join: {sorted(missing_val)}")

def build_text(df: pd.DataFrame) -> pd.Series:
    name = df["Name"].fillna("").astype(str)
    eng = df["English name"].fillna("").astype(str)
    syn = df["Synopsis"].fillna("").astype(str)
    eng = eng.where(eng.str.upper() != "UNKNOWN", "")
    return (name + " \n" + eng + " \n" + syn).str.strip()

train_df = train_df.copy()
val_df = val_df.copy()
train_df["text"] = build_text(train_df)
val_df["text"] = build_text(val_df)

def parse_genres(s: str) -> list[str]:
    if pd.isna(s):
        return []
    raw = str(s).strip()
    if not raw:
        return []
    return [g.strip() for g in raw.split(",") if g.strip()]

train_df["genres_list"] = train_df["Genres"].apply(parse_genres)
val_df["genres_list"] = val_df["Genres"].apply(parse_genres)

# Build label vocabulary from training only
all_genres = sorted({g for row in train_df["genres_list"] for g in row})
if len(all_genres) == 0:
    raise ValueError("No genres found in training data.")

from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer(classes=all_genres)
Y_train = mlb.fit_transform(train_df["genres_list"]).astype(np.int64)
Y_val = mlb.transform(val_df["genres_list"]).astype(np.int64)

train_df["labels"] = list(Y_train)
val_df["labels"] = list(Y_val)

from datasets import Dataset, DatasetDict
train_ds = Dataset.from_pandas(train_df[["anime_id", "text", "labels"]], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[["anime_id", "text", "labels"]], preserve_index=False)

label_names = list(mlb.classes_)
num_labels = len(label_names)

dataset = DatasetDict({"train": train_ds, "validation": val_ds})
print(dataset)
print("num_labels:", num_labels)
print("labels:", label_names)

In [ ]:
# Quick exploration
print("Train size:", len(dataset["train"]))
print("Val size:", len(dataset["validation"]))

# Example
ex = dataset["train"][0]
print("Example keys:", ex.keys())
print("anime_id:", ex["anime_id"])
print("text preview:\n", ex["text"][:500])

# Label statistics
label_counts = Y_train.sum(axis=0)
label_stats = pd.DataFrame({"genre": label_names, "count_in_train": label_counts}).sort_values("count_in_train", ascending=False)
display(label_stats.head(10))
display(label_stats.tail(10))

# Avg labels per example
labels_per_row = Y_train.sum(axis=1)
print("Avg labels/example (train):", float(labels_per_row.mean()))
print("Min labels/example (train):", int(labels_per_row.min()))
print("Max labels/example (train):", int(labels_per_row.max()))


---
## 1.2 Fine-tuning Pretrained Models (#3.1)

Select **two** different pretrained transformer models (e.g. BERT, RoBERTa, DistilBERT, DeBERTa, ModernBERT, ELECTRA, T5, …).  
The models must **not** already be fine-tuned on your specific task.

### Fine-tuned Model 1

In [ ]:
# TODO: Load tokenizer and model
MODEL_1_NAME = "" 

tokenizer_1 = None
model_1 = None 

In [ ]:
# TODO: Tokenize the dataset for Model 1


In [ ]:
# TODO: Define training arguments and Trainer, then fine-tune Model 1


In [ ]:
# TODO: Evaluate Model 1 on the test split


### Fine-tuned Model 2

In [ ]:
# TODO: Load tokenizer and model
MODEL_2_NAME = ""

tokenizer_2 = None
model_2 = None

In [ ]:
# TODO: Tokenize the dataset for Model 2


In [ ]:
# TODO: Fine-tune Model 2


In [ ]:
# TODO: Evaluate Model 2 on the test split


### Discussion — Fine-tuned Models (10%)

Answer all of the following in this cell:

- Describe the models you selected and what steps you needed to perform to fine-tune them for your task. 

- List the size of each model in number of parameters and discuss the dataset that the model was pretrained on, as well as the compute requirements used for pretraining (if details are available, you may need to do some searching to find the answer), e.g., 10 Tesla RTX 8000 GPUs were trained for 24 hours

*(Write your discussion here)*

---
## 1.3 Zero-shot Classification (#3.2)

Use **two** different pretrained language models in a zero-shot setting (no additional training).  
Try multiple prompts on a small portion of the **training** data to identify the best prompt, then evaluate on the **test** set using only that prompt.

If you are using Llama.cpp, include the file in your submission and enter your filename here: *(filename)*

And include instructions on how to run the code: *(instructions)*

Otherwise, complete the following cells.

### Zero-shot Model 1 — Prompt Engineering

In [ ]:
# TODO: Load your first zero-shot model
ZS_MODEL_1_NAME = ""


In [ ]:
# TODO: Experiment with different prompts on a small sample of training data
# Document at least 3 prompt variants you tried

prompt_variants = [
    # "Prompt variant 1: ...",
    # "Prompt variant 2: ...",
    # "Prompt variant 3: ...",
]

# TODO: Run each prompt on training sample and record results


In [ ]:
# TODO: Run the BEST prompt on the full test set and compute evaluation metrics
BEST_PROMPT_ZS1 = ""  # your best prompt


### Zero-shot Model 2 — Prompt Engineering

In [ ]:
# TODO: Load your second zero-shot model
ZS_MODEL_2_NAME = ""


In [ ]:
# TODO: Experiment with prompt variants for Model 2


In [ ]:
# TODO: Evaluate best prompt on the test set
BEST_PROMPT_ZS2 = ""


### Discussion — Zero-shot Classification (10%)

Answer all of the following:

- Talk about the models used, size, pretraining data, and required compute to pretrain. 
- Describe how you prompted the models in order to get it to solve your classification task. 
- What other prompts did you consider and why do you think they didn’t work well?

*(Write your discussion here)*

---
## 1.4 Baselines (#3.3)

### Bag-of-Words Classifier

In [ ]:
# BoW baseline: TF-IDF + One-vs-Rest Logistic Regression (multi-label)
X_train_text = np.array(train_df["text"].tolist(), dtype=object)
X_val_text = np.array(val_df["text"].tolist(), dtype=object)

y_train = Y_train.astype(int)
y_val = Y_val.astype(int)

vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    stop_words="english",
)
X_train_vec = vectorizer.fit_transform(X_train_text)
X_val_vec = vectorizer.transform(X_val_text)

bow_clf = OneVsRestClassifier(
    LogisticRegression(
        max_iter=2000,
        solver="liblinear",
        class_weight=None,
    )
)

bow_clf.fit(X_train_vec, y_train)

# Probabilities per label
val_prob_bow = bow_clf.predict_proba(X_val_vec)

best_t_bow = best_threshold_on_val(y_true=y_val, y_prob=val_prob_bow, metric="micro_f1")
metrics_bow = multilabel_metrics(y_true=y_val, y_prob=val_prob_bow, threshold=best_t_bow)
metrics_bow["threshold"] = best_t_bow
print_metrics("BoW (TF-IDF + OvR LogisticRegression) [val]", metrics_bow)


### Simple Baselines: Majority Class & Random

In [ ]:
# Majority (singular-genre) baseline: always predict the most frequent *single* genre in training
# This matches the competition's described majority baseline.
train_label_counts = y_train.sum(axis=0)
majority_genre_idx = int(np.argmax(train_label_counts))

majority_pred = np.zeros_like(y_val)
majority_pred[:, majority_genre_idx] = 1
metrics_majority = multilabel_metrics(y_true=y_val, y_pred=majority_pred)
metrics_majority["majority_genre"] = label_names[majority_genre_idx]
print_metrics("Majority (single genre) [val]", metrics_majority)

# Random baseline: sample each label independently using its empirical prevalence in training
# (Not perfect, but a reasonable multi-label random baseline.)
label_prevalence = (y_train.mean(axis=0)).clip(0.0, 1.0)
rng = np.random.default_rng(SEED)
random_pred = (rng.random(size=y_val.shape) < label_prevalence).astype(int)

# Ensure at least 1 label per instance to avoid all-zero predictions dominating
empty_rows = random_pred.sum(axis=1) == 0
if empty_rows.any():
    random_pred[empty_rows, majority_genre_idx] = 1

metrics_random = multilabel_metrics(y_true=y_val, y_pred=random_pred)
print_metrics("Random (per-label prevalence) [val]", metrics_random)


### Discussion — Baselines (5%)

- Describe the setup for your baselines. 
- Include enough detail that someone else could reproduce your approach based on reading this section.

*(Write your discussion here)*

---
## 1.5 Results Summary

In [ ]:
# TODO: Populate this results table with your actual numbers

results = {
    "Model": [
        f"Fine-tuned: {MODEL_1_NAME}",
        f"Fine-tuned: {MODEL_2_NAME}",
        f"Zero-shot: {ZS_MODEL_1_NAME}",
        f"Zero-shot: {ZS_MODEL_2_NAME}",
        "BoW Classifier",
        "Majority Class Baseline",
        "Random Baseline",
    ],
    "Accuracy": [None, None, None, None, None, None, None],
    # Add other metrics columns as appropriate for your task:
    # "F1 (Macro)": [...],
    # "F1 (Weighted)": [...],
    # "Precision": [...],
    # "Recall": [...],
}

results_df = pd.DataFrame(results)
results_df

### Discussion — Results (15%)

Write a substantive analysis of your results.

- Show a table that includes all of your results. In the end, you should have 7 rows. Two fine-tuned models, two zero-shot models, one BoW approach, and two simple baselines (random and majority class). Include one column for each evaluation metric that you computed.

- Describe what observations and analysis you have from these results. Do not just repeat the information that readers can directly find in your table.

- Think about what these results mean. For example, if you had to make recommendations to someone else who wanted to build a model for this task, what would you say to them that would be important for them to consider?

*(Write your discussion here)*

---
# PART 2 — Self-Chosen Secondary Task

For this task, you only need to complete **either** #3.1 (fine-tuning) **or** #3.2 (zero-shot) — not both, and not the baselines.  

**Secondary task chosen:** *(name it here)*

## 2.1 Secondary Task Dataset

In [ ]:
# TODO: Load and briefly explore the secondary task dataset


## 2.2 Approach — Fine-tuning OR Zero-shot

*(Complete only one of the two sections below. Delete the one you are not doing.)*

### Option A: Fine-tuning (#3.1)
*(Complete this section if you chose to fine-tune models on the secondary task)*

In [ ]:
# TODO: Fine-tune one or two models on the secondary task
# Follow the same steps as Part 1


### Option B: Zero-shot (#3.2)
*(Complete this section if you chose to do zero-shot classification on the secondary task)*

In [ ]:
# TODO: Run one or two zero-shot models on the secondary task
# Follow the same steps as Part 1


## 2.3 Secondary Task Results

In [ ]:
# TODO: Display evaluation metrics for your secondary task models


### Discussion — Secondary Task (10%)

1. Describe the steps you took for the second task that you chose. 
2. Did you fine-tune models, perform zero-shot classification, or both? 
3. How did these models perform compared to the baselines provided?

*(Write your discussion here)*

---
## Reflection (10%)

Answer all of the following thoughtfully:

1. What did you learn during the completion of this assignment? 
2. What was unexpected, if anything? 
3. What challenges did you face and how did you overcome them?

*(Write your reflection here)*

---
## Code Attribution

List any code snippets, tutorials, or resources you referenced below. Include URLs and a short description of what was adapted.

| Source | URL | What was adapted |
|--------|-----|------------------|
| HuggingFace Trainer tutorial | https://huggingface.co/docs/transformers/training | Fine-tuning loop scaffold |
| *(add more rows as needed)* | | |